# 02 — Run Baselines
Runs all 5 baseline strategies and saves results to `../Results/`

In [1]:
import sys
import time
import numpy as np
import pandas as pd

from functions.data_pipeline import build_dataset, run_lookahead_tests
from functions.baseline import run_all_baselines, compute_all_metrics

## 1. Load Data

In [2]:
dataset = build_dataset("../Data/Outputs/Filtered/Data")
run_lookahead_tests(dataset)

DATA PIPELINE — RL Portfolio Allocation

[1/5] Loading raw data...
  Prices: (18692, 144)
  Mask:   (27240, 152)
  QQQ:    (1249, 1)
  VIX:    (1249, 1)

[2/5] Cleaning & aligning...
  Tickers:       144
  Trading dates: 1239
  Date range:    2021-03-01 → 2026-02-18
  Hourly bars:   7434
  Bars/day:      6
  Close coverage (members): 98.9% mean, 97.0% min

[3/5] Building per-asset features (cross-sectionally ranked)...
  Features per asset:  7
  Feature names:       ['intraday_ret', 'intraday_rvol', 'ret_1d', 'ret_20d', 'ret_5d', 'rvol_20d', 'rvol_5d']
  Total columns:       1008

[4/5] Building global features...
  Global features: ['vix_level', 'vix_change_5d', 'market_ret_1d', 'market_rvol_5d', 'dow']

[5/5] Applying warmup period (20 days for rolling windows)...
  Valid date range: 2021-03-29 → 2026-02-18
  Valid days:       1219
  Per-asset NaN:    31.087%
  Global NaN:       0.000%

PIPELINE COMPLETE

LOOK-AHEAD BIAS TESTS

  Test 1: Feature-future return correlation check
    re

## 2. Run All Baselines (Full Period)

In [3]:
t0 = time.time()

all_results = run_all_baselines(
    dataset,
    start_date=None,   # uses full valid range
    end_date=None,
    transaction_cost_bps=5.0,
    results_dir="../Results",
    tag="full",
    verbose=True,
)

print(f"\nTotal time: {time.time() - t0:.1f}s")


RUNNING ALL BASELINES
  Period: start → end
  Transaction cost: 5.0 bps one-way
  Results dir: /Users/kamilkashif/Documents/University/Masters Thesis/Mater_Thesis/Results

  ▶ QQQ Buy-and-Hold... Done — IR2: 0.2817, ARC: 14.9%, MDD: 35.1%

  ▶ Equal-Weight Monthly... Done — IR2: 0.0979, ARC: 8.0%, MDD: 32.6%

  ▶ Inverse Volatility... Done — IR2: 0.0997, ARC: 8.1%, MDD: 32.2%

  ▶ Momentum Top-20%... Done — IR2: 0.0332, ARC: 5.4%, MDD: 38.6%

  ▶ Supervised + MVO... Done — IR2: 0.0000, ARC: -3.8%, MDD: 44.9%

  💾 Saved: ../Results/equity_curves_full.csv
  💾 Saved: ../Results/daily_returns_full.csv
  💾 Saved: ../Results/turnover_full.csv
  💾 Saved: ../Results/performance_metrics_full.csv

STRATEGY                     ARC%    ASD%    MDD%    MLD     IR1     IR2     TO%
  QQQ Buy-and-Hold          14.9   22.5   35.1  1.95  0.663 0.2817   0.00
  Inverse Volatility         8.1   20.4   32.2  2.08  0.397 0.0997   1.56
  Equal-Weight Monthly       8.0   20.2   32.6  2.08  0.397 0.0979   0.37

## 3. Walk-Forward Folds
Run baselines on each test fold separately for fair comparison with RL agent later.

In [4]:
# Walk-forward fold definitions
# Train periods are for RL training — baselines just run on test periods
FOLDS = {
    "fold1": {"test_start": "2024-01-01", "test_end": "2024-06-30"},
    "fold2": {"test_start": "2024-07-01", "test_end": "2025-02-28"},
    "fold3": {"test_start": "2025-03-01", "test_end": "2026-02-18"},
}

fold_results = {}

for fold_name, dates in FOLDS.items():
    print(f"\n{'='*70}")
    print(f"FOLD: {fold_name} — Test: {dates['test_start']} → {dates['test_end']}")
    print(f"{'='*70}")
    
    fold_results[fold_name] = run_all_baselines(
        dataset,
        start_date=dates["test_start"],
        end_date=dates["test_end"],
        transaction_cost_bps=5.0,
        results_dir="../Results",
        tag=f"{fold_name}_test",
        verbose=True,
    )


FOLD: fold1 — Test: 2024-01-01 → 2024-06-30

RUNNING ALL BASELINES
  Period: 2024-01-01 → 2024-06-30
  Transaction cost: 5.0 bps one-way
  Results dir: /Users/kamilkashif/Documents/University/Masters Thesis/Mater_Thesis/Results

  ▶ QQQ Buy-and-Hold... Done — IR2: 18.7520, ARC: 44.8%, MDD: 7.1%

  ▶ Equal-Weight Monthly... Done — IR2: 2.0369, ARC: 13.6%, MDD: 7.0%

  ▶ Inverse Volatility... Done — IR2: 1.9509, ARC: 13.3%, MDD: 7.0%

  ▶ Momentum Top-20%... Done — IR2: 5.3228, ARC: 29.5%, MDD: 8.8%

  ▶ Supervised + MVO... Done — IR2: 0.2021, ARC: 5.5%, MDD: 11.0%

  💾 Saved: ../Results/equity_curves_fold1_test.csv
  💾 Saved: ../Results/daily_returns_fold1_test.csv
  💾 Saved: ../Results/turnover_fold1_test.csv
  💾 Saved: ../Results/performance_metrics_fold1_test.csv

STRATEGY                     ARC%    ASD%    MDD%    MLD     IR1     IR2     TO%
  QQQ Buy-and-Hold          44.8   15.1    7.1  0.15  2.976 18.7520   0.00
  Momentum Top-20%          29.5   18.6    8.8  0.21  1.583 5.3228

## 4. Quick Summary

In [5]:
# Load saved metrics for verification
print("\n=== Full Period Metrics ===")
metrics_full = pd.read_csv("../Results/performance_metrics_full.csv", index_col=0)
display(metrics_full)

for fold_name in FOLDS:
    print(f"\n=== {fold_name} Test Metrics ===")
    m = pd.read_csv(f"../Results/performance_metrics_{fold_name}_test.csv", index_col=0)
    display(m)


=== Full Period Metrics ===


,Absolute Return (%),ARC (%),ASD (%),Max Drawdown (%),MLD (years),IR1,IR2,Avg Daily Turnover (%),Total TC (%)
Strategy,,,,,,,,,
QQQ Buy-and-Hold,97.3545,14.9126,22.4795,35.1187,1.9482,0.6634,0.2817,0.0000,0.0000
Equal-Weight Monthly,46.0309,8.0377,20.2418,32.5878,2.0831,0.3971,0.0979,0.3664,0.2231
Inverse Volatility,46.5132,8.0979,20.3985,32.2320,2.0831,0.3970,0.0997,1.5647,0.9529
Momentum Top-20%,29.9352,5.3805,22.5546,38.6482,2.1545,0.2386,0.0332,34.7167,21.1424
Supervised + MVO,-16.4294,-3.7547,22.9542,44.9105,4.1939,0.0000,0.0000,68.1085,41.4781



=== fold1 Test Metrics ===


,Absolute Return (%),ARC (%),ASD (%),Max Drawdown (%),MLD (years),IR1,IR2,Avg Daily Turnover (%),Total TC (%)
Strategy,,,,,,,,,
QQQ Buy-and-Hold,19.3537,44.7917,15.0515,7.1083,0.1468,2.9759,18.7520,0.0000,0.0000
Equal-Weight Monthly,6.4109,13.6343,13.0138,7.0126,0.3293,1.0477,2.0369,0.2690,0.0165
Inverse Volatility,6.2415,13.2763,12.9495,6.9771,0.2738,1.0252,1.9509,1.1928,0.0734
Momentum Top-20%,13.5274,29.5043,18.6441,8.7718,0.2103,1.5825,5.3228,28.9480,1.7803
Supervised + MVO,2.8235,5.5404,13.8462,10.9717,0.3135,0.4001,0.2021,56.8571,3.4967



=== fold2 Test Metrics ===


,Absolute Return (%),ARC (%),ASD (%),Max Drawdown (%),MLD (years),IR1,IR2,Avg Daily Turnover (%),Total TC (%)
Strategy,,,,,,,,,
QQQ Buy-and-Hold,5.7654,6.4021,20.4127,13.5577,0.3333,0.3136,0.1481,0.0000,0.0000
Equal-Weight Monthly,3.9411,6.9133,16.5492,10.7021,0.3214,0.4177,0.2699,0.3168,0.0258
Inverse Volatility,4.5061,7.8159,16.5329,10.7469,0.3214,0.4727,0.3438,1.4388,0.1173
Momentum Top-20%,-12.8736,-18.3853,21.0995,16.5335,0.6150,0.0000,0.0000,33.7006,2.7466
Supervised + MVO,4.0373,7.9649,19.5513,10.7469,0.3214,0.4074,0.3019,49.4477,4.0300



=== fold3 Test Metrics ===


,Absolute Return (%),ARC (%),ASD (%),Max Drawdown (%),MLD (years),IR1,IR2,Avg Daily Turnover (%),Total TC (%)
Strategy,,,,,,,,,
QQQ Buy-and-Hold,22.4952,22.7814,23.3620,16.9976,0.1865,0.9751,1.3070,0.0000,0.0000
Equal-Weight Monthly,13.9362,14.1204,20.9346,16.3435,0.1865,0.6745,0.5828,0.3760,0.0449
Inverse Volatility,13.9031,14.0133,21.0933,16.1721,0.1865,0.6644,0.5757,1.5000,0.1792
Momentum Top-20%,29.1240,29.6422,23.1324,15.0764,0.1905,1.2814,2.5194,30.3908,3.6317
Supervised + MVO,6.7361,6.0896,24.0836,16.1721,0.1865,0.2529,0.0952,74.4566,8.8976


In [6]:
print("\nFiles saved in ../Results/:")
import os
for f in sorted(os.listdir("../Results")):
    if f.endswith(".csv"):
        size = os.path.getsize(f"../Results/{f}") / 1024
        print(f"  {f:<45} {size:>6.1f} KB")


Files saved in ../Results/:
  daily_returns_fold1_test.csv                    14.4 KB
  daily_returns_fold2_test.csv                    19.0 KB
  daily_returns_fold3_test.csv                    27.9 KB
  daily_returns_full.csv                         141.4 KB
  equity_curves_fold1_test.csv                    12.7 KB
  equity_curves_fold2_test.csv                    16.8 KB
  equity_curves_fold3_test.csv                    24.6 KB
  equity_curves_full.csv                         124.9 KB
  performance_metrics_fold1_test.csv               0.5 KB
  performance_metrics_fold2_test.csv               0.5 KB
  performance_metrics_fold3_test.csv               0.5 KB
  performance_metrics_full.csv                     0.5 KB
  turnover_fold1_test.csv                         11.7 KB
  turnover_fold2_test.csv                         15.5 KB
  turnover_fold3_test.csv                         22.6 KB
  turnover_full.csv                              114.7 KB
